# SHARP vs Doppler-ResAT on the same LOEO folds

Fold 1 holds out the bedroom and trains on S6a, S6b, S7a; Fold 2 holds out the living room and trains on the nine bedroom sessions plus S7a. Within the training sessions each activity stream is split in time (75% training, 25% validation) with windows cut separately on each side; 5 classes (E, L, W, R, J). Doppler-ResAT uses your trained fold models when they are attached (folders `fold_1/`, `fold_2/` with `model.keras` and `norm_stats.pkl`), otherwise it is retrained with the same settings. SHARP's classifier (`network_utility.py`) is trained with its own recipe on the same split, and both are tested on the same held-out windows (stride 17). Input: the `doppler_traces` dataset.

Doppler-ResAT on Fold 2 needs about 17 h on a T4, more than Kaggle's 12 h limit. Its training state is saved after every epoch and the run stops before the limit; run the notebook again with the previous output attached and training continues from the last saved epoch. The tables include every model finished so far.

Turn on a GPU. To continue in a new session, attach the previous `results` folder as a dataset: finished runs are skipped and included in the tables. Everything is written to `/kaggle/working/results` and zipped to `results.zip`.

## Colab setup (run this first)

Continues the LOEO run from Kaggle: SHARP folds 1–2 and Doppler-ResAT fold 1 are finished, and Doppler-ResAT fold 2 continues from its saved epoch.

1. **Runtime → Change runtime type → T4 GPU.**
2. Open the **🔑 Secrets** panel on the left, add a secret named `KAGGLE_API_TOKEN` with your Kaggle token as its value, and turn on notebook access.
3. **Runtime → Run all.** Allow Google Drive access when asked.

Results are written to `MyDrive/doppler_resat_loeo` after every epoch. If Colab disconnects, reconnect and **Run all** again: training continues from the last saved epoch.

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
!pip install -q --upgrade kaggle
!mkdir -p /kaggle/input /kaggle/working /content/drive/MyDrive/doppler_resat_loeo
!ln -sfn /content/drive/MyDrive/doppler_resat_loeo /kaggle/working/results
![ -d /kaggle/input/doppler-traces ] || kaggle datasets download gyanendramohanpatel/doppler-traces -p /kaggle/input/doppler-traces --unzip
![ -d /kaggle/input/loeo-run1 ] || kaggle kernels output gyanendramohanpatel/doppler-resat-vs-sharp-loeo -p /kaggle/input/loeo-run1

In [ ]:
FOLDS = [1, 2]
SEED = 0
HELD_OUT = {1: 'bedroom', 2: 'living room'}
CLASS_NAMES = ['Empty', 'Sitting', 'Walking', 'Running', 'Jumping']

In [ ]:
import os, glob, pickle, itertools, shutil, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from scipy.ndimage import uniform_filter1d
from scipy.special import softmax
from sklearn.metrics import recall_score, f1_score, confusion_matrix


def find_root(pattern):
    hits = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
    return os.path.dirname(os.path.dirname(hits[0])) if hits else None


DOPPLER_DIR = find_root('S1a/S1a_E_stream_0.txt')
OUT = '/kaggle/working/results'
os.makedirs(OUT, exist_ok=True)
GPUS = tf.config.list_physical_devices('GPU')
for gpu in GPUS:
    tf.config.experimental.set_memory_growth(gpu, True)

WINDOW, NUM_ANTENNAS, BATCH_SIZE, EPOCHS = 340, 4, 32, 25
# each model trains with its own recipe; both are tested on the same windows
RECIPES = {'Doppler-ResAT': {'stride': 17, 'logits': False},  # ISED configuration, unchanged
           'SHARP': {'stride': 30, 'logits': True}}           # SHARP's CSI_doppler_create_dataset_train.py + CSI_network.py
MODELS = list(RECIPES)
CLASSES = {5: ['E', 'L', 'W', 'R', 'J'], 8: ['E', 'L', 'S', 'C', 'W', 'R', 'J', 'H']}
ISED_RECORDINGS = ['E', 'L', 'W', 'R', 'J', 'J1', 'J2']  # recordings the ISED LOEO runs used
SMOOTH_ACROSS_RECORDINGS = True  # as the ISED fold scripts; False smooths each recording separately
BEDROOM = ['S1a', 'S1b', 'S1c', 'S2a', 'S2b', 'S3a', 'S4a', 'S4b', 'S5a']
LOEO = {1: (['S6a', 'S6b', 'S7a'], BEDROOM), 2: (BEDROOM + ['S7a'], ['S6a', 'S6b'])}  # (train, test)

## Models: Doppler-ResAT (ISED) and SHARP (`network_utility.py`)

In [ ]:
def residual_block(x, filters, name_prefix='res'):
    shortcut = x
    x = tf.keras.layers.Conv2D(filters, (3, 3), padding='same', use_bias=False, name=f'{name_prefix}_conv1')(x)
    x = tf.keras.layers.BatchNormalization(name=f'{name_prefix}_bn1')(x)
    x = tf.keras.layers.ReLU(name=f'{name_prefix}_relu1')(x)
    x = tf.keras.layers.Conv2D(filters, (3, 3), padding='same', use_bias=False, name=f'{name_prefix}_conv2')(x)
    x = tf.keras.layers.BatchNormalization(name=f'{name_prefix}_bn2')(x)
    if shortcut.shape[-1] != filters:
        shortcut = tf.keras.layers.Conv2D(filters, (1, 1), padding='same', use_bias=False,
                                          name=f'{name_prefix}_proj_conv')(shortcut)
        shortcut = tf.keras.layers.BatchNormalization(name=f'{name_prefix}_proj_bn')(shortcut)
    x = tf.keras.layers.Add(name=f'{name_prefix}_add')([x, shortcut])
    return tf.keras.layers.ReLU(name=f'{name_prefix}_relu2')(x)


def channel_attention(x, reduction_ratio=8, name_prefix='ca'):
    channels = x.shape[-1]
    dense_down = tf.keras.layers.Dense(max(1, channels // reduction_ratio), activation='relu', use_bias=False,
                                       name=f'{name_prefix}_dense_down')
    dense_up = tf.keras.layers.Dense(channels, use_bias=False, name=f'{name_prefix}_dense_up')
    avg_pool = tf.keras.layers.GlobalAveragePooling2D(name=f'{name_prefix}_avg_pool')(x)
    avg_pool = tf.keras.layers.Reshape((1, 1, channels), name=f'{name_prefix}_avg_reshape')(avg_pool)
    max_pool = tf.keras.layers.GlobalMaxPooling2D(name=f'{name_prefix}_max_pool')(x)
    max_pool = tf.keras.layers.Reshape((1, 1, channels), name=f'{name_prefix}_max_reshape')(max_pool)
    scale = tf.keras.layers.Activation('sigmoid', name=f'{name_prefix}_sigmoid')(
        dense_up(dense_down(avg_pool)) + dense_up(dense_down(max_pool)))
    return tf.keras.layers.Multiply(name=f'{name_prefix}_mul')([x, scale])


def spatial_attention(x, kernel_size=7, name_prefix='sa'):
    avg_pool = tf.keras.layers.Lambda(lambda t: tf.reduce_mean(t, axis=-1, keepdims=True),
                                      name=f'{name_prefix}_avg_pool')(x)
    max_pool = tf.keras.layers.Lambda(lambda t: tf.reduce_max(t, axis=-1, keepdims=True),
                                      name=f'{name_prefix}_max_pool')(x)
    concat = tf.keras.layers.Concatenate(axis=-1, name=f'{name_prefix}_concat')([avg_pool, max_pool])
    attn_map = tf.keras.layers.Conv2D(1, kernel_size, padding='same', use_bias=False, name=f'{name_prefix}_conv')(concat)
    attn_map = tf.keras.layers.Activation('sigmoid', name=f'{name_prefix}_sigmoid')(attn_map)
    return tf.keras.layers.Multiply(name=f'{name_prefix}_mul')([x, attn_map])


def cbam_block(x, reduction_ratio=8, spatial_kernel=7, name_prefix='cbam'):
    x = channel_attention(x, reduction_ratio, name_prefix=f'{name_prefix}_ch')
    return spatial_attention(x, spatial_kernel, name_prefix=f'{name_prefix}_sp')


def build_sharp_plus_plus(input_shape, num_classes, lstm_units=128, dense_units=256, dropout_rate=0.5):
    inputs = tf.keras.Input(shape=input_shape, name='spectrogram_input')
    x = tf.keras.layers.Conv2D(64, (3, 3), padding='same', use_bias=False, name='stem_conv')(inputs)
    x = tf.keras.layers.BatchNormalization(name='stem_bn')(x)
    x = tf.keras.layers.ReLU(name='stem_relu')(x)
    x = residual_block(x, 64, name_prefix='res1')
    x = tf.keras.layers.MaxPooling2D((2, 2), name='pool1')(x)
    x = residual_block(x, 128, name_prefix='res2')
    x = tf.keras.layers.MaxPooling2D((2, 2), name='pool2')(x)
    x = residual_block(x, 256, name_prefix='res3')
    x = cbam_block(x, reduction_ratio=8, spatial_kernel=7)
    x = tf.keras.layers.Reshape((-1, x.shape[2] * x.shape[3]), name='reshape_lstm')(x)
    x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(lstm_units, return_sequences=True, dropout=0.2,
                                                           recurrent_dropout=0.0), name='bilstm_1')(x)
    x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(lstm_units, return_sequences=True, dropout=0.2,
                                                           recurrent_dropout=0.0), name='bilstm_2')(x)
    x = tf.keras.layers.GlobalAveragePooling1D(name='gap')(x)
    x = tf.keras.layers.Dense(dense_units, activation='relu', name='dense_embed')(x)
    x = tf.keras.layers.Dropout(dropout_rate, name='dropout')(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax', dtype='float32', name='output')(x)
    return tf.keras.Model(inputs=inputs, outputs=outputs, name='SHARP_plus_plus')


def get_optimizer(lr, weight_decay):
    try:
        return tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay)
    except AttributeError:
        return tf.keras.optimizers.experimental.AdamW(learning_rate=lr, weight_decay=weight_decay)


def conv2d_bn(x_in, filters, kernel_size, strides=(1, 1), padding='same', activation='relu', bn=False, name=None):
    x = tf.keras.layers.Conv2D(filters, kernel_size, strides=strides, padding=padding, name=name)(x_in)
    if bn:
        bn_name = None if name is None else name + '_bn'
        x = tf.keras.layers.BatchNormalization(axis=3, name=bn_name)(x)
    if activation is not None:
        x = tf.keras.layers.Activation(activation)(x)
    return x


def reduction_a_block_small(x_in, base_name):
    x1 = tf.keras.layers.MaxPool2D((2, 2), strides=(2, 2), padding='valid')(x_in)
    x2 = conv2d_bn(x_in, 5, (2, 2), strides=(2, 2), padding='valid', name=base_name + 'conv2_1_res_a')
    x3 = conv2d_bn(x_in, 3, (1, 1), name=base_name + 'conv3_1_res_a')
    x3 = conv2d_bn(x3, 6, (2, 2), name=base_name + 'conv3_2_res_a')
    x3 = conv2d_bn(x3, 9, (4, 4), strides=(2, 2), padding='same', name=base_name + 'conv3_3_res_a')
    return tf.keras.layers.Concatenate()([x1, x2, x3])


def csi_network_inc_res(input_sh, output_sh):
    x_input = tf.keras.Input(input_sh)
    x2 = reduction_a_block_small(x_input, base_name='1st')
    x3 = conv2d_bn(x2, 3, (1, 1), name='conv4')
    x = tf.keras.layers.Flatten()(x3)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Dense(output_sh, activation=None, name='dense2')(x)
    return tf.keras.Model(inputs=x_input, outputs=x, name='csi_model')

## Data, training and inference

In [ ]:
def recordings(root, session, classes, ised=False):
    # (label, name) of each recording; the label is the first letter of the activity, as in SHARP's dataset scripts
    names = sorted({f[len(session) + 1:].split('_stream_')[0] for f in os.listdir(f'{root}/{session}') if '_stream_' in f})
    if ised:  # the ISED fold scripts' order: class by class, then J, J1, J2
        return [(classes.index(c), n) for c in classes for n in ISED_RECORDINGS if n[0] == c and n in names]
    return [(classes.index(n[0]), n) for n in names if n[0] in classes]


def load_recording(root, session, name):
    # (time, 100, antennas), mean-subtracted along time
    streams = [pickle.load(open(f'{root}/{session}/{session}_{name}_stream_{a}.txt', 'rb')) for a in range(NUM_ANTENNAS)]
    return np.stack([s - s.mean(axis=0, keepdims=True) for s in streams], axis=-1).astype(np.float16)


def load_sessions(root, sessions, classes, ised=False):
    # all recordings in one array, plus (offset, length, label) of each recording
    data, recs, offset = [], [], 0
    for s in sessions:
        for label, name in recordings(root, s, classes, ised):
            x = load_recording(root, s, name)
            data.append(x)
            recs.append((offset, len(x), label))
            offset += len(x)
    return np.concatenate(data), recs


def window_index(recs, lo, hi, stride):
    # (start, label, antenna) of the windows inside the time split [lo, hi) of each recording
    return np.array([(o + t, label, ant) for o, T, label in recs for t in range(int(T * lo), int(T * hi) - WINDOW + 1, stride)
                     for ant in range(NUM_ANTENNAS)], dtype=np.int32).reshape(-1, 3)


def compute_norm_stats(giant_array, indices, window_length, max_samples=5000):
    n = min(max_samples, len(indices))
    sample_idxs = indices[np.random.choice(len(indices), n, replace=False)]
    samples = np.stack([giant_array[start:start + window_length, :, ant:ant + 1] for (start, _, ant) in sample_idxs])
    samples = samples.astype(np.float64)
    mean = samples.mean(axis=(0, 1), keepdims=True)[0].astype(np.float32)
    std = np.maximum(samples.std(axis=(0, 1), keepdims=True), 1e-8)[0].astype(np.float32)
    return mean, std


def make_dataset(master, idx, mean, std, num_classes, shuffle, augment):
    master_t, mean_t, std_t = tf.constant(master), tf.constant(mean), tf.constant(std)
    ds = tf.data.Dataset.from_tensor_slices(idx)
    if shuffle:
        ds = ds.shuffle(min(len(idx), 20000), reshuffle_each_iteration=True)
    ds = ds.repeat()

    def window(i):
        x = (tf.cast(master_t[i[0]:i[0] + WINDOW, :, i[2]:i[2] + 1], tf.float32) - mean_t) / std_t
        return tf.ensure_shape(x, (WINDOW, master.shape[1], 1)), tf.one_hot(i[1], depth=num_classes)

    def augment_fn(x, label):
        x = x + tf.random.normal(shape=tf.shape(x), mean=0.0, stddev=0.05)

        def mask_time():
            win_len = tf.shape(x)[0]
            mask_size = tf.random.uniform([], 1, 30, dtype=tf.int32)
            mask_start = tf.random.uniform([], 0, win_len - mask_size, dtype=tf.int32)
            t = tf.range(win_len)
            mask = tf.logical_or(t < mask_start, t >= mask_start + mask_size)
            return x * tf.reshape(tf.cast(mask, tf.float32), [win_len, 1, 1])
        return tf.cond(tf.random.uniform([]) < 0.3, mask_time, lambda: x), label

    ds = ds.map(window, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


def set_policy(name):
    use_mixed = name == 'Doppler-ResAT' and GPUS  # ISED trained in mixed precision; SHARP in float32
    tf.keras.mixed_precision.set_global_policy('mixed_float16' if use_mixed else 'float32')


class Resume(tf.keras.callbacks.Callback):
    # saves the training state after every epoch and stops before the deadline; a later session continues from it
    KEYS = ['best', 'wait', 'best_epoch', 'best_weights', 'cooldown_counter']  # EarlyStopping, ReduceLROnPlateau

    def __init__(self, path, others, deadline, mean, std, state):
        super().__init__()
        self.path, self.others, self.deadline, self.mean, self.std, self.state = path, others, deadline, mean, std, state
        self.interrupted = False

    def on_train_begin(self, logs=None):
        # runs after the other callbacks have reset themselves
        if self.state:
            for cb, attrs in zip(self.others, self.state['callbacks']):
                for k, v in attrs.items():
                    setattr(cb, k, v)
        self.t = time.time()

    def on_epoch_end(self, epoch, logs=None):
        opt = self.model.optimizer
        state = {'epoch': epoch + 1, 'weights': self.model.get_weights(), 'optimizer': [v.numpy() for v in opt.variables],
                 'learning_rate': float(np.array(opt.learning_rate)), 'mean': self.mean, 'std': self.std,
                 'callbacks': [{k: getattr(cb, k) for k in self.KEYS if hasattr(cb, k)} for cb in self.others]}
        with open(self.path, 'wb') as fp:
            pickle.dump(state, fp)
        now = time.time()
        if not self.model.stop_training and epoch + 1 < EPOCHS and now + 1.2 * (now - self.t) > self.deadline:
            self.model.stop_training = self.interrupted = True
            print(f'Epoch {epoch + 1}: stopping before the time limit; the training state is saved to {self.path}')
        self.t = now


def train_model(name, master, recs, splits, num_classes, seed, resume=None):
    # one model, trained with its own recipe on the (train, val) time splits of the recordings;
    # resume=(file, deadline) saves the state every epoch, stops before the deadline and continues from an attached file
    train_idx, val_idx = (window_index(recs, lo, hi, RECIPES[name]['stride']) for lo, hi in splits)
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    set_policy(name)
    state = find_result(resume[0]) if resume else None
    if name == 'SHARP':
        mean, std = np.zeros((1, master.shape[1], 1), np.float32), np.ones((1, master.shape[1], 1), np.float32)
        model = csi_network_inc_res((WINDOW, master.shape[1], 1), num_classes)
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                      loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
                      metrics=[tf.keras.metrics.CategoricalAccuracy(name='accuracy')])
        callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)]
    else:
        mean, std = (state['mean'], state['std']) if state else compute_norm_stats(master, train_idx, WINDOW)
        model = build_sharp_plus_plus((WINDOW, master.shape[1], 1), num_classes)
        model.compile(optimizer=get_optimizer(1e-4, 1e-4), loss=tf.keras.losses.CategoricalFocalCrossentropy(gamma=2.0),
                      metrics=[tf.keras.metrics.CategoricalAccuracy(name='accuracy')])
        callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
                     tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)]
    start = 0
    if state:
        model.set_weights(state['weights'])
        model.optimizer.build(model.trainable_variables)
        if len(model.optimizer.variables) != len(state['optimizer']):
            raise ValueError(f'{resume[0]} does not match this model')
        for v, value in zip(model.optimizer.variables, state['optimizer']):
            v.assign(value)
        model.optimizer.learning_rate = state['learning_rate']
        start = state['epoch']
        print(f'{name}: continuing from epoch {start} of {resume[0]}')
    if resume:
        callbacks.append(Resume(f'{OUT}/{resume[0]}', list(callbacks), resume[1], mean, std, state))
    history = model.fit(make_dataset(master, train_idx, mean, std, num_classes, True, name != 'SHARP'),
                        epochs=EPOCHS, initial_epoch=start, steps_per_epoch=int(np.ceil(len(train_idx) / BATCH_SIZE)),
                        validation_data=make_dataset(master, val_idx, mean, std, num_classes, False, False),
                        validation_steps=int(np.ceil(len(val_idx) / BATCH_SIZE)), callbacks=callbacks, verbose=2)
    if resume and callbacks[-1].interrupted:
        return None
    if resume:
        os.remove(f'{OUT}/{resume[0]}')
    # Doppler-ResAT restores its best epoch; SHARP keeps its last epoch, as CSI_network.py does
    epoch = callbacks[0].best_epoch + 1 if name != 'SHARP' else start + len(history.history['loss'])
    return model, mean, std, epoch


def predict_recording(model, x, lo, stride, mean, std):
    # per-antenna outputs (windows, antennas, classes) from fraction lo to the end of one recording
    starts = range(int(len(x) * lo), len(x) - WINDOW + 1, stride)
    w = (np.stack([x[t:t + WINDOW] for t in starts]).astype(np.float32) - mean) / std
    w = np.moveaxis(w, -1, 1).reshape(-1, WINDOW, x.shape[1], 1)
    return model.predict(w, batch_size=4 * BATCH_SIZE, verbose=0).reshape(len(starts), NUM_ANTENNAS, -1)


def predict_sets(model, root, sets, classes, lo, stride, mean, std, ised=False):
    # {set: [(outputs, labels) for each recording]}; lo gives the start fraction per set (default 0)
    out = {}
    for name, sessions in sets.items():
        out[name] = []
        for s in sessions:
            for label, rec in recordings(root, s, classes, ised):
                P = predict_recording(model, load_recording(root, s, rec), lo.get(name, 0), stride, mean, std)
                out[name].append((P, np.full(len(P), label)))
    return out


def find_result(fname):
    # results of earlier sessions can be attached as a Kaggle dataset; they are copied into OUT
    if not os.path.exists(f'{OUT}/{fname}'):
        hits = glob.glob(f'/kaggle/input/**/{fname}', recursive=True)
        if not hits:
            return None
        shutil.copy(hits[0], OUT)
    with open(f'{OUT}/{fname}', 'rb') as fp:
        return pickle.load(fp)


def save_result(fname, result):
    with open(f'{OUT}/{fname}', 'wb') as fp:
        pickle.dump(result, fp)

## Metrics (protocol code; smoothing as in the ISED fold scripts; SHARP's vote fusion)

In [ ]:
def smooth(p, k=11):
    # p: (T, C) fused probabilities in time order
    return uniform_filter1d(p, size=k, axis=0)


def sharp_vote(Z):
    # SHARP's decision fusion (CSI_network_test.py): majority of the antenna decisions, summed logits break ties
    pred = Z.sum(axis=1).argmax(axis=1)
    for i, v in enumerate(Z.argmax(axis=2)):
        lab, cnt = np.unique(v, return_counts=True)
        if len(lab) == 1 or (len(lab) == 2 and cnt[0] != cnt[1]):
            pred[i] = lab[cnt.argmax()]
    return pred


def evaluate(streams, classes, antennas=(0, 1, 2, 3), use_smoothing=True, logits=False, vote=False):
    # streams: list of (P, y); P: (T, 4, C) per-antenna softmax (logits for SHARP), y: (T,) labels
    y_true = np.concatenate([y for _, y in streams])
    if vote:
        y_pred = np.concatenate([sharp_vote(P) for P, _ in streams])
    else:
        fused = [(softmax(P, axis=-1) if logits else P)[:, list(antennas), :].mean(axis=1) for P, _ in streams]
        if use_smoothing:
            fused = [smooth(np.concatenate(fused))] if SMOOTH_ACROSS_RECORDINGS else [smooth(f) for f in fused]
        y_pred = np.concatenate(fused).argmax(axis=1)
    labels = list(range(len(classes)))
    present = [c for c in labels if c in set(y_true)]  # S2 has no S, C or H recordings
    rec = recall_score(y_true, y_pred, labels=present, average=None) * 100
    return {
        "per_class_recall": dict(zip([classes[c] for c in present], np.round(rec, 2))),
        "mean_recall": round(float(rec.mean()), 2),
        "macro_f1": round(float(f1_score(y_true, y_pred, labels=present, average="macro")), 3),
        "accuracy": round(float(100 * (y_true == y_pred).mean()), 2),
        "confusion": confusion_matrix(y_true, y_pred, labels=labels),
    }


def flat(e):
    return {'mean_recall': e['mean_recall'], 'macro_f1': e['macro_f1'], 'accuracy': e['accuracy'],
            **{f'recall_{c}': v for c, v in e['per_class_recall'].items()}}


def antenna_subsets(streams, classes, n, **kw):
    # {metric: (mean, std)} over all combinations of n antennas
    runs = [flat(evaluate(streams, classes, s, **kw)) for s in itertools.combinations(range(NUM_ANTENNAS), n)]
    return {k: (float(np.mean([r[k] for r in runs])), float(np.std([r[k] for r in runs]))) for k in runs[0]}


def confusion_frame(cm, classes, **tags):
    # raw window counts, rows = true class, columns = predicted class
    df = pd.DataFrame(cm, index=pd.Index(classes, name='true'), columns=classes).reset_index()
    return df.assign(**tags)[list(tags) + list(df.columns)]


# rows reported per model: (model, label, fusion)
VARIANTS = [('Doppler-ResAT', 'Doppler-ResAT', {}),
            ('SHARP', 'SHARP re-run (probability fusion)', {}),
            ('SHARP', 'SHARP re-run (vote fusion, no smoothing)', {'vote': True})]

In [ ]:
STRIDE_LOEO_TEST = 17  # ISED test stride
DEADLINE = time.time() + 11.25 * 3600  # Kaggle stops a run at 12 h; training is saved and stopped before that


def run_loeo(fold, name):
    # one model on one LOEO fold; the trained ISED fold model is reused when attached
    fname = f'loeo_{name}_fold{fold}.pkl'
    if find_result(fname):
        return
    classes = CLASSES[5]
    train_sessions, test_sessions = LOEO[fold]
    ised = glob.glob(f'/kaggle/input/**/fold_{fold}/model.keras', recursive=True)
    if name == 'Doppler-ResAT' and ised:
        tf.keras.backend.clear_session()
        set_policy(name)
        model = build_sharp_plus_plus((WINDOW, 100, 1), len(classes))
        model.load_weights(ised[0])
        with open(os.path.join(os.path.dirname(ised[0]), 'norm_stats.pkl'), 'rb') as fp:
            norm = pickle.load(fp)
        mean, std, epoch = norm['mean'], norm['std'], 'ISED model'
    else:
        master, recs = load_sessions(DOPPLER_DIR, train_sessions, classes, ised=True)
        trained = train_model(name, master, recs, [(0, 0.75), (0.75, 1)], len(classes), SEED,
                              resume=(f'{fname[:-4]}.resume.pkl', DEADLINE))
        del master
        if trained is None:
            print(f'{name}, fold {fold}: not finished in this session. Run the notebook again with this output attached.')
            return
        model, mean, std, epoch = trained
        model.save_weights(f'{OUT}/{fname[:-4]}.weights.h5')
    probs = predict_sets(model, DOPPLER_DIR, {'test': test_sessions}, classes, {}, STRIDE_LOEO_TEST, mean, std,
                         ised=True)['test']
    save_result(fname, {'model': name, 'fold': fold, 'probs': probs, 'logits': RECIPES[name]['logits'],
                        'epoch': epoch, 'test_stride': STRIDE_LOEO_TEST})

## Train and test

In [ ]:
for name in ['SHARP', 'Doppler-ResAT']:  # Doppler-ResAT fold 2 (~17 h) runs last and continues over two sessions
    for fold in FOLDS:
        run_loeo(fold, name)

## Results

In [ ]:
classes = CLASSES[5]
rows, confusions, windows = [], [], []
for fold in FOLDS:
    for name, label, kw in VARIANTS:
        r = find_result(f'loeo_{name}_fold{fold}.pkl')
        if r is None:
            continue
        e = evaluate(r['probs'], classes, logits=r['logits'], **kw)
        raw = evaluate(r['probs'], classes, logits=r['logits'], use_smoothing=False, **kw)
        rows.append({'Fold': fold, 'Held out': HELD_OUT[fold], 'Model': label, 'Epoch used': r['epoch'], **flat(e),
                     **{f'{k} [no smoothing]': v for k, v in flat(raw).items()}})
        confusions.append(confusion_frame(e['confusion'], classes, Fold=fold, Model=label))
        if name == 'Doppler-ResAT':
            y = np.concatenate([y for _, y in r['probs']])
            windows.append({'Fold': fold, 'Test stride': r['test_stride'], **dict(zip(classes, np.bincount(y, minlength=5)))})

tables = {'comparison': pd.DataFrame(rows), 'confusion': pd.concat(confusions), 'test_windows': pd.DataFrame(windows)}
for key, df in tables.items():
    df.to_csv(f'{OUT}/loeo_{key}.csv', index=False)
print('4-antenna fusion with smoothing (no smoothing in the [no smoothing] columns); recall and accuracy in %')
for df in tables.values():
    display(df)

# per-class recall in the held-out rooms
results = tables['comparison']
fig, axes = plt.subplots(1, len(FOLDS), figsize=(6 * len(FOLDS), 3.5), squeeze=False, sharey=True)
for ax, fold in zip(axes[0], FOLDS):
    r = results[results['Fold'] == fold].reset_index(drop=True)
    width = 0.8 / len(r)
    for k, row in r.iterrows():
        ax.bar(np.arange(len(classes)) + (k - (len(r) - 1) / 2) * width, [row[f'recall_{c}'] for c in classes], width,
               label=row['Model'])
    ax.set(xticks=range(len(classes)), xticklabels=CLASS_NAMES, ylim=(0, 100), title=f'Fold {fold}: {HELD_OUT[fold]} held out')
axes[0][0].set_ylabel('Recall (%)')
axes[0][0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(f'{OUT}/loeo_per_class_recall.png', dpi=200)
fig.savefig(f'{OUT}/loeo_per_class_recall.pdf')
plt.show()

In [ ]:
!cd /kaggle/working && zip -qr results.zip results

In [ ]:
!cp /kaggle/working/results.zip /content/drive/MyDrive/doppler_resat_loeo_results.zip